###Transform Races Data

1. Read bronze races table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (raceName -> race_name, raceId -> race_id)
4. Rename columns to make them more meaningful (date -> race_date)
5. Remove duplicate records
6. Transform values of columns race_name to Title Case
8. Write the transformed data to silver races table

Below changes are required to implement incremental Load Processing

1. Accept batch_id as a parameter to the notebook
2. Process data for only the batch_id being passed in (i.e., filter reading from bronze using the batch_id)
3. Add created_timestamp, updated_timestamp and batch_id to the silver table.
4. Merge the processed data to the silver table
    - created_timestamp should only be populated at the time of the inserting/creating the record. It should not be updated during the merge update.
    - Ensure that we are not overwriting the data in silver table by older brinze data (re-run scenario)

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

#####Step 1 - Read bronze races table

In [0]:
# races_df = spark.read.option('versionAsOf', '0').table(bronze_table)

In [0]:
races_df = (
    spark.table(bronze_table)
        .filter(F.col("batch_id") == v_batch_id)
)

#####Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
from pyspark.sql import functions as F

In [0]:
races_selected_df = races_df.select(F.col("season"), F.col("round"), F.col("raceName"), F.col("date"), F.col("circuitId"), F.col("ingestion_timestamp"), F.col("source_file"))

Step 3 & 4 - Standardise Column Names

 - Standardise column names using snake_case (circuit -> circuit_id, raceName -> race_name)
 - Rename columns to make them more meaningful (date -> race_date)

In [0]:
# races_renamed_df = (
#     races_selected_df
#     .withColumnRenamed("circuitId", "circuit_id")
#     .withColumnRenamed("raceName", "race_name")
#     .withColumnRenamed("date", "race_date")
# )

In [0]:
races_renamed_df = (
    races_selected_df
        .withColumnsRenamed({
           "circuitId": "circuit_id",
            "raceName": "race_name",
            "date": "race_date"
            })
)

#####Step 5 - Remove duplicate records

In [0]:
# races_distinct_df = races_valid_df.distinct()

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season", "round"])

In [0]:
display(races_distinct_df)

#####Step 6 - Transform values of columns race_name to Title Case

In [0]:
races_final_df = (races_distinct_df
                     .withColumn('race_name', F.initcap(F.col("race_name"))
                    ))

In [0]:
display(races_final_df)

#####Step 7 - Write the transformed data to silver races table

In [0]:
from delta.tables import DeltaTable

races_to_write_df = (
    races_final_df
        .withColumn("batch_id", F.lit(v_batch_id))
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
)

if not spark.catalog.tableExists(silver_table):
    (
        races_to_write_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
    )
else:
    existing_columns = spark.table(silver_table).columns
    missing_columns = []

    if "batch_id" not in existing_columns:
        missing_columns.append("batch_id STRING")
    if "created_timestamp" not in existing_columns:
        missing_columns.append("created_timestamp TIMESTAMP")
    if "updated_timestamp" not in existing_columns:
        missing_columns.append("updated_timestamp TIMESTAMP")

    if missing_columns:
        spark.sql(f"ALTER TABLE {silver_table} ADD COLUMNS ({', '.join(missing_columns)})")

    delta_table = DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
            .merge(
                races_to_write_df.alias("s"),
                "t.season = s.season AND t.round = s.round"
            )
            .whenMatchedUpdate(
                condition="t.batch_id IS NULL OR s.batch_id >= t.batch_id",
                set={
                    "race_name": "s.race_name",
                    "race_date": "s.race_date",
                    "circuit_id": "s.circuit_id",
                    "ingestion_timestamp": "s.ingestion_timestamp",
                    "source_file": "s.source_file",
                    "batch_id": "s.batch_id",
                    "updated_timestamp": "s.updated_timestamp"
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
    )

In [0]:
display(spark.table(silver_table))